# NAIP and canopy height for the Elkinsville NE mapping module

Produces one four-band NAIP mosaic and one canopy height model per imagery date
over the AIS photo-interpretation module in Brown County, Indiana, all on a
single analysis grid, so that a stand can be measured the same way at every
date and differences between dates need no resampling.

Runtime: **Runtime > Change runtime type > T4 GPU**. The canopy height model is
PyTorch, so a CPU runtime will run but will take hours instead of minutes.

Steps: configure, install, mount Drive, fetch the conditioning rasters, fetch
NAIP per year, run inference per year, put every output on the analysis grid,
verify the alignment, export.

The heavy pieces are cached on Drive, so re-running after a disconnect skips
whatever is already done.

## 1. Configuration

In [ ]:
from pathlib import Path

SITE = "ElkinsvilleNE"

# Imagery dates. Planetary Computer carries this module from 2012 onward.
# 2012 and 2014 are 1.0 m; 2016 onward are 0.6 m.
YEARS = [2012, 2014, 2016, 2018, 2020, 2022]
SOURCE = "pc"                 # "pc" = Planetary Computer, "gee" = Earth Engine
EE_PROJECT = "dyce-biomass"   # only used when SOURCE == "gee"

# The analysis grid. Bounds are the module footprint snapped outward to a 3 m
# multiple, which is a whole number of pixels at both 0.6 m and 1.0 m so the
# two grids nest. Inference runs on a 150 m buffer of this, because the canopy
# height model blends chips and its outermost ~50 m is unreliable.
ANALYSIS_BOUNDS = (559425.0, 4323936.0, 564888.0, 4330917.0)   # EPSG:26916
INFER_BOUNDS = (559275.0, 4323786.0, 565038.0, 4331067.0)
STAC_BBOX_4326 = (-86.3125, 39.0625, -86.25, 39.1250)
DST_CRS = "EPSG:26916"
ANALYSIS_RES_M = 0.6

# Native ground sample distance per year. Inference runs at the native
# resolution and only the output is warped, so the model sees the input
# statistics it was trained on.
NATIVE_RES_M = {2012: 1.0, 2014: 1.0, 2016: 0.6, 2018: 0.6, 2020: 0.6, 2022: 0.6}

DRIVE_FOLDER = "CSDV_Elkinsville"
NAIPCHM_REPO = Path("/content/naip-chm")
MODEL_CHECKPOINT = "model/model_20251016.pt"
MODEL_CONFIG = "configs/config.yaml"
CHIP_SIZE = 432
CHIP_OVERLAP = 0.2

print(f"{SITE}: {len(YEARS)} dates, source={SOURCE}")

## 2. Check the runtime

Fail here rather than three hours into a CPU run.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or
      "No GPU detected. Set Runtime > Change runtime type > T4 GPU and rerun.")

## 3. Install

Clone the canopy height model and install its dependencies. The model weights,
about 88 MB, ship inside the repository, so there is nothing else to download
for the network itself.

**After this cell finishes, restart the runtime** (Runtime > Restart session),
then run cell 1 again and continue from cell 4. The install replaces Colab's
preinstalled PyTorch, and continuing without a restart leaves a half-loaded
CUDA extension that fails in confusing ways at inference time.

In [ ]:
%%bash
set -e
if [ ! -d /content/naip-chm ]; then
  git clone --depth 1 https://github.com/smorf-ntsg/naip-chm.git /content/naip-chm
fi
cd /content/naip-chm
pip install -q -r requirements.txt
pip install -q rio-cogeo pystac-client planetary-computer geopandas
echo
echo "Installed. Now restart the runtime, rerun cell 1, and continue from cell 4."

Optionally install this project's library too, which brings in the stand
metric code and the fetch helper used below. Without it the notebook falls back
to an inline copy of the same two functions.

In [ ]:
%%bash
set -e
if [ ! -d /content/csdv ]; then
  echo "Clone or copy the CSDV repository to /content/csdv to use csdv_core here."
else
  pip install -q -e /content/csdv
fi

## 4. Mount Drive and lay out the cache

The conditioning rasters are about 1.65 GB and the NAIP mosaics are a few
hundred megabytes each. Caching them on Drive means a disconnected session
resumes instead of starting over.

In [ ]:
import os, shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
DRIVE_COND = DRIVE_ROOT / "conditioning_data"
DRIVE_NAIP = DRIVE_ROOT / "naip_mosaic"
DRIVE_CHM_RAW = DRIVE_ROOT / "chm_raw"
DRIVE_CHM_GRID = DRIVE_ROOT / "chm_grid"
for d in (DRIVE_COND, DRIVE_NAIP, DRIVE_CHM_RAW, DRIVE_CHM_GRID):
    d.mkdir(parents=True, exist_ok=True)

# Point the repo's conditioning directory at Drive so the download happens once.
local_cond = NAIPCHM_REPO / "data" / "conditioning_data"
local_cond.parent.mkdir(parents=True, exist_ok=True)
if local_cond.is_symlink():
    local_cond.unlink()
elif local_cond.exists():
    shutil.rmtree(local_cond)
os.symlink(DRIVE_COND, local_cond)

LOCAL_NAIP = Path("/content/work/naip"); LOCAL_NAIP.mkdir(parents=True, exist_ok=True)
LOCAL_CHM = Path("/content/work/chm"); LOCAL_CHM.mkdir(parents=True, exist_ok=True)
print("Drive cache:", DRIVE_ROOT)

## 5. Conditioning rasters

Five static layers the model conditions on: elevation, a climate principal
component stack, a soil principal component stack, land cover and ecoregion.
Downloaded once, then read from the Drive cache.

In [ ]:
REQUIRED = ["elevation.tif", "climate_pca.tif", "soil_pca.tif", "nlcd.tif", "ecoregion.tif"]

missing = [name for name in REQUIRED if not (DRIVE_COND / name).exists()]
if missing:
    print("Downloading:", missing)
    !cd {NAIPCHM_REPO} && echo n | python scripts/download_conditioning_data.py
else:
    print("Conditioning rasters already cached.")

still_missing = [name for name in REQUIRED if not (DRIVE_COND / name).exists()]
assert not still_missing, f"Conditioning rasters missing: {still_missing}"
for name in REQUIRED:
    print(f"  {name:16s} {(DRIVE_COND / name).stat().st_size / 1e6:8.1f} MB")

## 6. Fetch NAIP

Each year's quads are read through a warped virtual raster already targeted at
that year's grid, so the mosaic step is a paste and every pixel is resampled
once. Access tokens expire after about an hour, so the catalogue is searched
inside the loop rather than once up front.

Filenames carry an eight-digit date because the model reads day of year from
the filename. Where a year's quads span more than one flight the median date is
used for the whole mosaic, and every individual date goes into the manifest.

In [ ]:
import json
import numpy as np

try:
    from csdv_core.download.naip_pc import grid_from_bounds, median_date_tag, naip_mosaic, search_naip_items
    print("Using csdv_core.download.naip_pc")
except ImportError:
    print("csdv_core not installed; using the inline fallback")
    from affine import Affine
    from datetime import datetime
    import planetary_computer, pystac_client, rasterio
    from rasterio.enums import Resampling
    from rasterio.vrt import WarpedVRT

    class _Grid(tuple):
        @property
        def transform(self): return self[0]
        @property
        def width(self): return self[1]
        @property
        def height(self): return self[2]

    def grid_from_bounds(bounds, resolution):
        minx, miny, maxx, maxy = bounds
        w = int(round((maxx - minx) / resolution))
        h = int(round((maxy - miny) / resolution))
        return _Grid((Affine(resolution, 0, minx, 0, -resolution, maxy), w, h))

    def median_date_tag(values):
        parsed = sorted(datetime.fromisoformat(str(v).replace("Z", "+00:00")) for v in values)
        return parsed[len(parsed) // 2].strftime("%Y%m%d")

    def search_naip_items(bbox, *, year, catalog=None):
        catalog = catalog or pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=planetary_computer.sign_inplace)
        items = list(catalog.search(collections=["naip"], bbox=list(bbox),
                                    datetime=f"{year}-01-01/{year}-12-31").items())
        if not items:
            raise ValueError(f"No NAIP for {year}")
        return items

    def naip_mosaic(items, out_path, *, dst_crs, dst_transform, dst_width, dst_height,
                    resampling="bilinear", compress="DEFLATE"):
        out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
        mosaic = np.zeros((4, dst_height, dst_width), dtype="uint8")
        written = np.zeros((dst_height, dst_width), dtype=bool)
        env = {"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
               "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
               "GDAL_HTTP_MAX_RETRY": "5", "GDAL_HTTP_RETRY_DELAY": "3"}
        with rasterio.Env(**env):
            for item in items:
                with rasterio.open(item.assets["image"].href) as src, WarpedVRT(
                        src, crs=dst_crs, transform=dst_transform, width=dst_width,
                        height=dst_height, resampling=Resampling[resampling]) as vrt:
                    block = vrt.read()
                covered = block.any(axis=0); fill = covered & ~written
                mosaic[:, fill] = block[:, fill]; written |= covered
        profile = dict(driver="GTiff", height=dst_height, width=dst_width, count=4,
                       dtype="uint8", crs=dst_crs, transform=dst_transform,
                       compress=compress, tiled=True, blockxsize=512, blockysize=512,
                       BIGTIFF="IF_SAFER")
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(mosaic)
        dates = [str(i.properties["datetime"]) for i in items]
        return out_path, {"n_items": len(items), "item_ids": [i.id for i in items],
                          "acq_dates": sorted(dates), "date_tag": median_date_tag(dates),
                          "gsd": sorted({i.properties.get("gsd") for i in items} - {None}),
                          "crs": str(dst_crs), "transform": list(dst_transform)[:6],
                          "shape": [dst_height, dst_width],
                          "coverage_fraction": float(written.mean()),
                          "path": str(out_path)}

In [ ]:
import pystac_client, planetary_computer

manifest = {}
naip_paths = {}

for year in YEARS:
    res = NATIVE_RES_M[year]
    grid = grid_from_bounds(INFER_BOUNDS, res)

    # Sign inside the loop: Planetary Computer tokens expire in about an hour.
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )
    items = search_naip_items(STAC_BBOX_4326, year=year, catalog=catalog)
    tag = median_date_tag([i.properties["datetime"] for i in items])
    name = f"NAIP_{SITE}_{tag}_{str(res).replace('.', 'p')}m.tif"

    cached = DRIVE_NAIP / name
    local = LOCAL_NAIP / name
    if cached.exists():
        if not local.exists():
            local.symlink_to(cached)
        print(f"{year}: cached {name}")
        side_car = DRIVE_NAIP / f"{name}.json"
        prov = json.loads(side_car.read_text()) if side_car.exists() else {
            "n_items": len(items), "item_ids": [i.id for i in items],
            "acq_dates": sorted(str(i.properties["datetime"]) for i in items),
            "date_tag": tag, "coverage_fraction": float("nan"),
            "gsd": sorted({i.properties.get("gsd") for i in items} - {None}),
        }
    else:
        print(f"{year}: mosaicking {len(items)} quads at {res} m "
              f"-> {grid.width} x {grid.height} px")
        _, prov = naip_mosaic(items, local, dst_crs=DST_CRS,
                              dst_transform=grid.transform,
                              dst_width=grid.width, dst_height=grid.height)
        shutil.copy2(local, cached)
        (DRIVE_NAIP / f"{name}.json").write_text(json.dumps(prov, indent=2))

    prov["native_res_m"] = res
    manifest[year] = prov
    naip_paths[year] = local
    print(f"     {prov['n_items']} quads, dates {sorted(set(d[:10] for d in prov['acq_dates']))}, "
          f"coverage {prov['coverage_fraction']:.4f}")

## 7. Canopy height inference

One run per date. The model chips the image at 432 pixels with 20 percent
overlap, samples the conditioning rasters at each chip centre, and blends the
chips back together. Output is a cloud-optimised GeoTIFF of canopy height in
centimetres.

If a run fails on the mosaic size, the fallback is to split `INFER_BOUNDS` into
two by two sub-tiles with 150 m of overlap, infer each, and mosaic the
interiors. The overlap has to exceed the roughly 50 m unreliable rim, or the
seams carry the artefact.

In [ ]:
import subprocess, time

chm_raw = {}
for year in YEARS:
    tag = manifest[year]["date_tag"]
    out_dir = LOCAL_CHM / f"{SITE}_{tag}"
    cached = list(DRIVE_CHM_RAW.glob(f"*{tag}*_chm.tif"))
    if cached:
        chm_raw[year] = cached[0]
        print(f"{year}: cached {cached[0].name}")
        continue

    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "scripts/inference.py",
        "--naip-quad", str(naip_paths[year]),
        "--output-dir", str(out_dir),
        "--model-checkpoint", MODEL_CHECKPOINT,
        "--config", MODEL_CONFIG,
        "--static-rasters-dir", "data/conditioning_data",
        "--chip-size", str(CHIP_SIZE),
        "--chip-overlap", f"{CHIP_OVERLAP:g}",
    ]
    started = time.time()
    result = subprocess.run(cmd, cwd=NAIPCHM_REPO, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:]); print(result.stderr[-4000:])
        raise RuntimeError(f"Inference failed for {year}")

    produced = sorted(out_dir.glob("*_chm.tif"))[-1]
    shutil.copy2(produced, DRIVE_CHM_RAW / produced.name)
    chm_raw[year] = produced
    print(f"{year}: {produced.name} in {time.time() - started:.0f} s")

## 8. Put every date on the analysis grid

The model returns canopy height in centimetres on the grid of its input, which
for 2012 and 2014 is the 1.0 m grid. Converting to metres and warping to the
shared 0.6 m analysis grid here means everything downstream compares like with
like, and the cropping drops the unreliable outer rim.

The 1.0 m dates keep a coarser effective resolution than the 0.6 m dates even
after they share a grid. `native_res_m` travels in the manifest so that any
figure can mark those dates rather than implying they are equivalent.

In [ ]:
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

grid = grid_from_bounds(ANALYSIS_BOUNDS, ANALYSIS_RES_M)
NODATA = -9999.0
chm_paths = {}

for year in YEARS:
    tag = manifest[year]["date_tag"]
    name = f"CHM_{SITE}_{tag}_0p6m.tif"
    cached = DRIVE_CHM_GRID / name
    if cached.exists():
        chm_paths[year] = cached
        print(f"{year}: cached {name}")
        continue

    with rasterio.open(chm_raw[year]) as src:
        raw = src.read(1).astype("float32")
        src_nodata = src.nodatavals[0]
        src_transform, src_crs = src.transform, src.crs
    if src_nodata is not None:
        raw = np.where(raw == np.float32(src_nodata), np.nan, raw)
    metres = raw * 0.01   # the model writes centimetres

    out = np.full((grid.height, grid.width), np.nan, dtype="float32")
    reproject(
        source=metres, destination=out,
        src_transform=src_transform, src_crs=src_crs, src_nodata=np.nan,
        dst_transform=grid.transform, dst_crs=DST_CRS, dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )
    out = np.where(np.isfinite(out), out, NODATA)

    profile = dict(driver="GTiff", height=grid.height, width=grid.width, count=1,
                   dtype="float32", crs=DST_CRS, transform=grid.transform,
                   nodata=NODATA, compress="DEFLATE", predictor=3, tiled=True,
                   blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
    local = LOCAL_CHM / name
    with rasterio.open(local, "w", **profile) as dst:
        dst.write(out, 1)
    shutil.copy2(local, cached)
    chm_paths[year] = cached
    valid = out != NODATA
    print(f"{year}: {name}  valid {valid.mean():.4f}  median {np.median(out[valid]):.1f} m")

The NAIP mosaics also need to sit on the analysis grid, because texture is
computed from the near infrared band inside the same stand mask as the canopy
height metrics.

In [ ]:
naip_grid_paths = {}
DRIVE_NAIP_GRID = DRIVE_ROOT / "naip_grid"; DRIVE_NAIP_GRID.mkdir(exist_ok=True)

for year in YEARS:
    tag = manifest[year]["date_tag"]
    name = f"NAIP_{SITE}_{tag}_0p6m.tif"
    cached = DRIVE_NAIP_GRID / name
    if cached.exists():
        naip_grid_paths[year] = cached
        print(f"{year}: cached {name}")
        continue

    with rasterio.open(naip_paths[year]) as src:
        block = src.read()
        src_transform, src_crs = src.transform, src.crs
    out = np.zeros((4, grid.height, grid.width), dtype="uint8")
    for b in range(4):
        reproject(
            source=block[b], destination=out[b],
            src_transform=src_transform, src_crs=src_crs,
            dst_transform=grid.transform, dst_crs=DST_CRS,
            resampling=Resampling.bilinear,
        )
    profile = dict(driver="GTiff", height=grid.height, width=grid.width, count=4,
                   dtype="uint8", crs=DST_CRS, transform=grid.transform,
                   compress="DEFLATE", tiled=True, blockxsize=512, blockysize=512,
                   BIGTIFF="IF_SAFER")
    local = LOCAL_NAIP / name
    with rasterio.open(local, "w", **profile) as dst:
        dst.write(out)
        dst.descriptions = ("red", "green", "blue", "nir")
    shutil.copy2(local, cached)
    naip_grid_paths[year] = cached
    print(f"{year}: {name}")

## 9. Verification gate

Do not export until this passes. A silent misalignment would leave every single
date looking reasonable while corrupting every difference between dates, which
is the part the trajectory classification depends on.

In [ ]:
reference = None
rows = []
for year in YEARS:
    with rasterio.open(chm_paths[year]) as src:
        current = (str(src.crs), tuple(src.transform)[:6], (src.height, src.width))
        arr = src.read(1, masked=True)
    if reference is None:
        reference = current
    assert current == reference, f"{year} is not on the analysis grid:\n{current}\n{reference}"

    with rasterio.open(naip_grid_paths[year]) as src:
        naip_grid = (str(src.crs), tuple(src.transform)[:6], (src.height, src.width))
    assert naip_grid == reference, f"NAIP {year} is not on the analysis grid"

    values = arr.compressed()
    assert values.min() >= -0.01, f"{year}: negative canopy height {values.min():.2f} m"
    assert values.max() <= 120.0, f"{year}: implausible canopy height {values.max():.1f} m"
    rows.append({
        "year": year,
        "native_res_m": NATIVE_RES_M[year],
        "date_tag": manifest[year]["date_tag"],
        "n_quads": manifest[year]["n_items"],
        "valid_fraction": float(1.0 - arr.mask.mean()),
        "median_height_m": float(np.median(values)),
        "gap_fraction": float((values < 2.0).mean()),
    })

import pandas as pd
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print("\nAll dates share one grid:", reference[2], reference[0])

A quick look at each date, with the stand polygons drawn over them. This is a
sanity check on registration and on whether the disturbances are where the
interpreters put them, not a result.

In [ ]:
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent

# Upload the AIS delivery to the Drive folder to get the stand outlines here.
GDB = str(DRIVE_ROOT / "Indiana-ElkinsvilleNE_revised.gdb")
try:
    import geopandas as gpd
    stands = gpd.read_file(GDB, layer="DisturbancePoly")
    stands = stands[stands["DistType"] != 999]
except Exception as exc:
    print("Stand polygons not available for the overlay:", exc)
    stands = None

fig, axes = plt.subplots(2, 3, figsize=(13, 11), constrained_layout=True)
for ax, year in zip(axes.ravel(), YEARS):
    with rasterio.open(chm_paths[year]) as src:
        arr = src.read(1, masked=True, out_shape=(src.height // 8, src.width // 8))
        extent = plotting_extent(src)
    im = ax.imshow(arr, cmap="viridis", vmin=0, vmax=30, extent=extent)
    if stands is not None:
        stands.boundary.plot(ax=ax, color="white", linewidth=0.4)
    ax.set_title(f"{year} ({NATIVE_RES_M[year]:g} m native)")
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.02, label="Canopy height (m)")
plt.show()

## 10. Manifest and export

The manifest is the provenance record: which quads went into each mosaic, every
acquisition date behind the single date tag, the native resolution, and the
grid everything landed on.

Copy the contents of the Drive folder into the repository under
`data/naip/ElkinsvilleNE/<year>/` and `data/naip_chm/ElkinsvilleNE/<year>/`, and
put the manifest at `data/naip_chm/ElkinsvilleNE/manifest.json`. The local
analysis notebook reads from there.

In [ ]:
import torch

for year in YEARS:
    manifest[year]["chm_path"] = str(chm_paths[year])
    manifest[year]["naip_grid_path"] = str(naip_grid_paths[year])
    manifest[year]["analysis_grid"] = {
        "crs": DST_CRS,
        "transform": list(grid.transform)[:6],
        "shape": [grid.height, grid.width],
        "bounds": list(ANALYSIS_BOUNDS),
        "resolution_m": ANALYSIS_RES_M,
    }
    manifest[year]["model_checkpoint"] = MODEL_CHECKPOINT
    manifest[year]["torch_version"] = torch.__version__
    manifest[year]["chip_size"] = CHIP_SIZE
    manifest[year]["chip_overlap"] = CHIP_OVERLAP

manifest_path = DRIVE_ROOT / "manifest.json"
manifest_path.write_text(json.dumps({str(k): v for k, v in manifest.items()}, indent=2))
print("Wrote", manifest_path)

total = sum(p.stat().st_size for p in list(DRIVE_CHM_GRID.glob("*.tif")) +
            list((DRIVE_ROOT / "naip_grid").glob("*.tif")))
print(f"Analysis-grid outputs: {total / 1e9:.2f} GB in {DRIVE_ROOT}")